In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
from pathlib import Path
import gc

# --- CONFIGURATION ---
ROOT = Path("C:/Users/Théo Lassale/Desktop/Perso/Stage/Stage_Malmö/IMU_LM_Data")
IN_PATH = ROOT / "data" / "merged_dataset" / "unified_dataset.parquet"
OUT_PATH = ROOT / "data" / "merged_dataset" / "filtered_3_labels.parquet"

# Les 3 labels que tu veux garder
LABELS_TO_KEEP = ['stationary_position','walk', 'rest_inactive']

# --- FILTRAGE PAR MORCEAUX ---
print(f" Filtrage du dataset pour garder : {LABELS_TO_KEEP}")

pf = pq.ParquetFile(IN_PATH)
writer = None
total_filtered = 0

# On parcourt le gros fichier par blocs de 1 million de lignes
for batch in pf.iter_batches(batch_size=1_000_000):
    # Conversion du morceau en DataFrame Pandas pour filtrer facilement
    df_chunk = batch.to_pandas()
    
    # Filtrage : on ne garde que les 3 labels
    mask = df_chunk['global_activity_label'].isin(LABELS_TO_KEEP)
    df_filtered = df_chunk[mask].copy()
    
    if len(df_filtered) > 0:
        # Re-conversion en Table PyArrow pour l'écriture
        table_filtered = pa.Table.from_pandas(df_filtered, schema=batch.schema)
        
        # Initialisation du fichier de sortie au premier morceau trouvé
        if writer is None:
            writer = pq.ParquetWriter(OUT_PATH, batch.schema, compression="zstd")
        
        writer.write_table(table_filtered)
        total_filtered += len(df_filtered)
    
    # Nettoyage mémoire
    del df_chunk, df_filtered
    gc.collect()

if writer:
    writer.close()

print(f"\n Terminé ! Nouveau dataset créé : {OUT_PATH}")
print(f"Nombre de lignes conservées : {total_filtered:,}")

# --- TEST DE CHARGEMENT RAPIDE ---
print("\nTentative de chargement du dataset filtré en RAM...")
try:
    df_final = pd.read_parquet(OUT_PATH)
    print(f"Succès ! Le dataset filtré pèse {df_final.memory_usage().sum() / 1e6:.2f} Mo en RAM.")
    print("Distribution des labels :")
    print(df_final['global_activity_label'].value_counts())
except Exception as e:
    print(f"Erreur lors du chargement : {e}")

 Filtrage du dataset pour garder : ['walk', 'rest_inactive']

 Terminé ! Nouveau dataset créé : C:\Users\Théo Lassale\Desktop\Perso\Stage\Stage_Malmö\IMU_LM_Data\data\merged_dataset\filtered_2_labels.parquet
Nombre de lignes conservées : 10,178,486

Tentative de chargement du dataset filtré en RAM...
Succès ! Le dataset filtré pèse 692.14 Mo en RAM.
Distribution des labels :
global_activity_label
rest_inactive    7224537
walk             2953949
Name: count, dtype: int64
